In [1]:
import sys
import torch
import platform

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

!nvidia-smi

Python version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch version: 2.11.0+cu128
CUDA available: True
CUDA version: 12.8
GPU: Tesla T4
Tue Sep 15 13:17:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             12W /   70W |       3MiB /  15360MiB |      0%     

In [ ]:
import os
import torch
import sys
import platform

# 1. Clone Repository (Safe to re-run)
if not os.path.exists("InfiniteTalk"):
    !git clone https://github.com/MeiGen-AI/InfiniteTalk
else:
    print("Repository already cloned.")

# 2. Enter Directory (Required after every restart)
%cd InfiniteTalk

# 3. Safe Dependency Installation
py_ver = f"cp{sys.version_info.major}{sys.version_info.minor}"
torch_ver = torch.__version__.split("+")[0]
torch_ver_short = ".".join(torch_ver.split(".")[:2])
cuda_ver = torch.version.cuda.replace(".", "")[:3] if torch.version.cuda else "cpu"
abi = "TRUE" if torch._C._GLIBCXX_USE_CXX11_ABI else "FALSE"

print(f"✅ Detected: Python={py_ver}, Torch={torch_ver}, CUDA={cuda_ver}, ABI={abi}")
print("Installing dependencies... (Fast mode enabled)")

# Install dependencies needed for extensions
!pip install --no-cache-dir packaging ninja psutil wheel

# Install Flash Attention 2 (Dynamic Wheel Selection)
try:
    import flash_attn
    print("✅ Flash Attention already installed.")
except ImportError:
    print("Installing Flash Attention...")

    # CASE 1: Python 3.12 + PyTorch 2.9 (User Verified Environment)
    if py_ver == "cp312" and torch_ver.startswith("2.9"):
        url = "https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.3/flash_attn-2.8.3+cu12torch2.9cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
    else:
        # CASE 2: Fallback to guessing official URL for other versions
        fa_v = "2.8.3"
        cu_tag = f"cu{cuda_ver[:2]}" # e.g., cu12
        wheel = f"flash_attn-{fa_v}+{cu_tag}torch{torch_ver_short}cxx11abi{abi}-{py_ver}-{py_ver}-linux_x86_64.whl"
        url = f"https://github.com/Dao-AILab/flash-attention/releases/download/v{fa_v}/{wheel}"

    try:
        print(f"Attempting to install: {url}")
        res = os.system(f"pip install --no-cache-dir {url}")
        if res != 0:
            raise Exception("Pip install returned non-zero exit code")
        print(f"✅ Successfully installed Flash Attention via wheel")
    except Exception as e:
        print(f"⚠️ Wheel install failed: {e}")
        print("Falling back to standard pip install (this may trigger a slow build)...")
        !pip install flash-attn --no-build-isolation

# Install other dependencies from requirements.txt
!sed -i 's/numpy>=1.23.5,<2/numpy>=1.23.5/' requirements.txt
!pip install --no-cache-dir -r requirements.txt
!pip install --no-cache-dir librosa huggingface_hub

# Install system dependencies
!apt-get update && apt-get install -y ffmpeg > /dev/null 2>&1

print("\n✅ Setup Complete! If you see a 'Restart Session' button, you can IGNORE it and try running the next cell.")

Cloning into 'InfiniteTalk'...
remote: Enumerating objects: 516, done.
remote: Counting objects: 100% (162/162), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 516 (delta 122), reused 80 (delta 80), pack-reused 354 (from 2)
Receiving objects: 100% (516/516), 241.41 MiB | 35.34 MiB/s, done.
Resolving deltas: 100% (189/189), done.
/content/InfiniteTalk
✅ Detected: Python=cp313, Torch=2.11.0, CUDA=128, ABI=TRUE
Installing dependencies... (Fast mode enabled)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 9.4 MB/s eta 0:00:00
Installing Flash Attention...
Attempting to install: https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.3/flash_attn-2.8.3+cu12torch2.11cxx11abiTRUE-cp313-cp313-linux_x86_64.whl
⚠️ Wheel install failed: Pip install returned non-zero exit code
Falling back to standard pip install (this may trigger a slow build)...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 88.0 MB/s eta 0:00:00
  Preparing metadata (setup.

In [ ]:
import os
os.makedirs("weights", exist_ok=True)
os.makedirs("weights/InfiniteTalk/quant_models", exist_ok=True)

# PREVENT DOUBLE DISK USAGE: Set HF cache to weights folder
os.environ["HF_HUB_CACHE"] = os.path.join(os.getcwd(), "weights/.cache")
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "true"

print("Downloading models (Optimized for space)...")

# 1. Download base model (Wan2.1-I2V-14B-480P)
!hf download Wan-AI/Wan2.1-I2V-14B-480P --local-dir ./weights/Wan2.1-I2V-14B-480P --local-dir-use-symlinks False

# 2. Download FP8 Quantized Weights (SAVED DISK & VRAM)
# These are smaller and required for Tesla T4 (16GB VRAM)
print("Downloading FP8 weights for Tesla T4 compatibility...")
!wget -O weights/InfiniteTalk/quant_models/infinitetalk_single_fp8.safetensors https://huggingface.co/MeiGen-AI/InfiniteTalk/resolve/main/quant_models/infinitetalk_single_fp8.safetensors
!wget -O weights/InfiniteTalk/quant_models/infinitetalk_multi_fp8.safetensors https://huggingface.co/MeiGen-AI/InfiniteTalk/resolve/main/quant_models/infinitetalk_multi_fp8.safetensors

# 3. Download English audio encoder
!hf download facebook/wav2vec2-base-960h --local-dir ./weights/wav2vec2-base-960h --local-dir-use-symlinks False

print("✅ Download Complete!")

In [ ]:
# Enable public link for Gradio
!sed -i 's/demo.launch(server_name="0.0.0.0", debug=True, server_port=8418)/demo.launch(server_name="0.0.0.0", debug=True, server_port=8418, share=True)/' app.py

print("🚀 Launching in FP8 Mode (Compatible with Tesla T4 16GB)")

# Launch Gradio interface with FP8 Optimization enabled
!python app.py \n    --ckpt_dir weights/Wan2.1-I2V-14B-480P \n    --wav2vec_dir 'weights/wav2vec2-base-960h' \n    --infinitetalk_dir weights/InfiniteTalk/single/infinitetalk.safetensors \n    --quant fp8 \n    --quant_dir weights/InfiniteTalk/quant_models/infinitetalk_single_fp8.safetensors \n    --num_persistent_param_in_dit 0 \n    --motion_frame 9